# Rabbit — Engine Dashboard (Colab)

Builds the C engine, creates a synthetic test model, and opens Rabbit's **own** dark dashboard:

- Chat panel
- Model stats
- Expert cache hit rate
- Expert activity grid

This is **not** Colibrì's UI.

Repo: [Stanley03-arch/Rabbit](https://github.com/Stanley03-arch/Rabbit)

## Cell 1 — Clone & build

In [ ]:
import os

os.chdir("/content")
!rm -rf Rabbit
!git clone --depth 1 https://github.com/Stanley03-arch/Rabbit.git
os.chdir("/content/Rabbit")

!make clean
!make 2>&1 | tail -25
!chmod +x rabbit tools/create_test_model 2>/dev/null; true
!make test-model

print("==== version ====")
!./rabbit version
print("==== files ====")
!ls -la rabbit models/ web/ web/static/

## Cell 2 — Install Flask & start dashboard

Colab cannot open `127.0.0.1` on your laptop. We use **`serve_kernel_port_as_iframe`** (not `as_window`).

Keep this cell running while you use the UI.

In [ ]:
!pip install -q flask

import subprocess
import sys
import time
import urllib.request
from google.colab import output

PORT = 7860
MODEL = "/content/Rabbit/models/tiny-test.rabbit"
REPO = "/content/Rabbit"

# Kill anything already on the port
subprocess.run(f"fuser -k {PORT}/tcp 2>/dev/null", shell=True)
time.sleep(1)

proc = subprocess.Popen(
    [
        sys.executable,
        "tools/rabbit_dashboard.py",
        "--model", MODEL,
        "--host", "0.0.0.0",
        "--port", str(PORT),
    ],
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print("Starting Rabbit dashboard...")
ready = False
for i in range(60):
    if proc.poll() is not None:
        out = proc.stdout.read() if proc.stdout else ""
        print(out)
        raise RuntimeError("Dashboard process exited early. See logs above.")

    # drain some log lines
    while proc.stdout and True:
        # non-blocking-ish read via short timeout loop
        break

    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1)
        ready = True
        print("Server is up on port", PORT)
        break
    except Exception:
        time.sleep(0.4)

if not ready:
    print("Server did not respond yet; opening proxy anyway.")

# Correct Colab embedding (do NOT use serve_kernel_port_as_window)
output.serve_kernel_port_as_iframe(PORT)
print("\nDashboard iframe should appear above.")
print("If it is blank, run Cell 3 for a clickable proxy URL.")

# Keep process alive for the lifetime of this cell if needed:
# (iframe keeps working while runtime is active)
try:
    while True:
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            print(line.rstrip())
        if proc.poll() is not None:
            print("Dashboard stopped.")
            break
        time.sleep(0.05)
except KeyboardInterrupt:
    proc.terminate()
    print("Stopped.")

## Cell 3 — Fallback: clickable proxy URL

Run this if the iframe is empty. Open the printed URL in a **new browser tab**.

In [ ]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(7860)")
print("Open this URL:")
print(url)

## What success looks like

| Area | Expected |
|------|----------|
| Left sidebar | Model name, layers, experts, quant |
| Cache | Hits / misses / hit rate bar after a message |
| Expert grid | Some cells warm/hot after generate |
| Chat | Reply appears (may look random on synthetic weights) |

**Do not** open `http://127.0.0.1:7860` or `https://localhost:7860` on your PC — those only work inside the Colab VM.

## Optional — larger int4 demo model

Run after Cell 1. Then restart the dashboard (Cell 2) with the new path.

In [ ]:
# Optional
!python tools/convert_hf_to_rabbit.py --demo --out models/demo-moe.rabbit --quant 4 --hidden-dim 128 --n-experts 16
!ls -lh models/demo-moe.rabbit
print("To use it: stop Cell 2, change MODEL path to models/demo-moe.rabbit, run Cell 2 again.")